In [ ]:
import os

os.makedirs('/kaggle/working/anime_cgan/data', exist_ok=True)
os.makedirs('/kaggle/working/anime_cgan/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/anime_cgan/results', exist_ok=True)
print("Dossiers créés")

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print(root)
    if files:
        print("  fichiers:", files[:3])  # affiche les 3 premiers fichiers

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", files[:3])
    print("------")

In [ ]:
import random
from PIL import Image
import os

RAW_DIR = '/kaggle/input/datasets/splcher/animefacedataset/images'
SAVE_DIR = '/kaggle/working/anime_cgan/data/images_5k'
os.makedirs(SAVE_DIR, exist_ok=True)

all_images = [f for f in os.listdir(RAW_DIR) if f.endswith('.jpg') or f.endswith('.png')]

random.seed(42)
selected = random.sample(all_images, 5000)

errors = 0
for fname in selected:
    try:
        img = Image.open(os.path.join(RAW_DIR, fname)).convert('RGB')
        img.resize((64, 64)).save(os.path.join(SAVE_DIR, fname))
    except:
        errors += 1

print(f"Images sauvegardées : {5000 - errors}/5000")
print(f"Erreurs : {errors}")

In [ ]:
import matplotlib.pyplot as plt

samples = random.sample(os.listdir(SAVE_DIR), 9)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, fname in zip(axes.flatten(), samples):
    ax.imshow(Image.open(os.path.join(SAVE_DIR, fname)))
    ax.axis('off')

plt.suptitle('Échantillon — 9 images du dataset')
plt.tight_layout()
plt.savefig('/kaggle/working/anime_cgan/results/dataset_sample.png')
plt.show()

In [ ]:
!pip install git+https://github.com/openai/CLIP.git -q
print(" CLIP installé")

In [ ]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device : {device}")

model_clip, preprocess_clip = clip.load("ViT-B/32", device=device)
print(" CLIP chargé")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

SAVE_DIR = '/kaggle/working/anime_cgan/data/images_5k'
image_files = sorted(os.listdir(SAVE_DIR))

all_embeddings = []
batch_size = 64

for i in range(0, len(image_files), batch_size):
    batch_files = image_files[i:i+batch_size]
    images = []
    for fname in batch_files:
        img = Image.open(os.path.join(SAVE_DIR, fname)).convert('RGB')
        images.append(preprocess_clip(img))
    
    images_tensor = torch.stack(images).to(device)
    
    with torch.no_grad():
        embeddings = model_clip.encode_image(images_tensor)
    
    all_embeddings.append(embeddings.cpu())
    
    if i % 500 == 0:
        print(f"  {i}/{len(image_files)} images traitées...")

all_embeddings = torch.cat(all_embeddings, dim=0)
torch.save({
    'embeddings': all_embeddings,
    'filenames': image_files
}, '/kaggle/working/anime_cgan/data/embeddings.pt')

print(f" Embeddings sauvegardés — shape : {all_embeddings.shape}")

In [ ]:
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, clip_dim=512, img_channels=3):
        super().__init__()
        
        self.fc = nn.Linear(z_dim + clip_dim, 512 * 4 * 4)
        
        self.net = nn.Sequential(
            # 4x4 → 8x8
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 8x8 → 16x16
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 16x16 → 32x32
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 32x32 → 64x64
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1),
            nn.Tanh()
        )
    
    def forward(self, z, clip_emb):
        x = torch.cat([z, clip_emb], dim=1)  # (batch, 612)
        x = self.fc(x)                        # (batch, 512*4*4)
        x = x.view(-1, 512, 4, 4)            # (batch, 512, 4, 4)
        return self.net(x)                    # (batch, 3, 64, 64)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, clip_dim=512, img_channels=3):
        super().__init__()
        
        self.conv = nn.Sequential(
            # 64x64 → 32x32
            nn.Conv2d(img_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 → 16x16
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 → 8x8
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 → 4x4
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        # fusion image + clip embedding
        self.fc = nn.Sequential(
            nn.Linear(512 * 4 * 4 + clip_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, clip_emb):
        x = self.conv(img)                          # (batch, 512, 4, 4)
        x = x.view(x.size(0), -1)                  # (batch, 512*4*4)
        x = torch.cat([x, clip_emb], dim=1)        # (batch, 512*4*4 + 512)
        return self.fc(x)                           # (batch, 1)

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

# test avec des tenseurs factices
z = torch.randn(8, 100).to(device)
clip_emb = torch.randn(8, 512).to(device)
fake_img = G(z, clip_emb)
score = D(fake_img, clip_emb)

print(f" Generator output : {fake_img.shape}")   # doit afficher (8, 3, 64, 64)
print(f" Discriminator output : {score.shape}")  # doit afficher (8, 1)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class AnimeDataset(Dataset):
    def __init__(self, img_dir, embeddings_path):
        data = torch.load(embeddings_path)
        self.embeddings = data['embeddings']  # (5000, 512)
        self.filenames = data['filenames']
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_dir, self.filenames[idx])).convert('RGB')
        return self.transform(img), self.embeddings[idx]

dataset = AnimeDataset(
    img_dir='/kaggle/working/anime_cgan/data/images_5k',
    embeddings_path='/kaggle/working/anime_cgan/data/embeddings.pt'
)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=2)
print(f" Dataset chargé : {len(dataset)} images, {len(dataloader)} batches")

In [ ]:
import torch.optim as optim

Z_DIM = 100
lr = 0.0002
beta1 = 0.5

G = Generator().to(device)
D = Discriminator().to(device)

opt_G = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))

criterion = nn.BCELoss()

# vecteurs z fixes pour suivre la progression visuellement
fixed_z = torch.randn(16, Z_DIM).to(device)
fixed_emb = dataset.embeddings[:16].to(device)

print(" Setup prêt")

In [ ]:
import torchvision.utils as vutils
import matplotlib.pyplot as plt

EPOCHS = 50
losses_G, losses_D = [], []

for epoch in range(EPOCHS):
    loss_G_epoch, loss_D_epoch = 0, 0

    for real_imgs, clip_embs in dataloader:
        real_imgs = real_imgs.to(device)
        clip_embs = clip_embs.to(device).float()
        batch_size = real_imgs.size(0)

        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # --- Entraîner Discriminator ---
        z = torch.randn(batch_size, Z_DIM).to(device)
        fake_imgs = G(z, clip_embs)

        loss_D = criterion(D(real_imgs, clip_embs), real_labels) + \
                 criterion(D(fake_imgs.detach(), clip_embs), fake_labels)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Entraîner Generator ---
        z = torch.randn(batch_size, Z_DIM).to(device)
        fake_imgs = G(z, clip_embs)
        loss_G = criterion(D(fake_imgs, clip_embs), real_labels)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        loss_G_epoch += loss_G.item()
        loss_D_epoch += loss_D.item()

    # moyennes
    avg_G = loss_G_epoch / len(dataloader)
    avg_D = loss_D_epoch / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f}")

    # sauvegarde checkpoint toutes les 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/G_epoch{epoch+1}.pth')
        torch.save(D.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/D_epoch{epoch+1}.pth')

        # grille d'images générées
        with torch.no_grad():
            fake_sample = G(fixed_z, fixed_emb)
        grid = vutils.make_grid(fake_sample, normalize=True, nrow=4)
        plt.figure(figsize=(8, 8))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis('off')
        plt.title(f'Epoch {epoch+1}')
        plt.savefig(f'/kaggle/working/anime_cgan/results/epoch_{epoch+1}.png')
        plt.show()

print(" Entraînement terminé !")

In [ ]:
import clip
import torch
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# charger CLIP
model_clip, preprocess_clip = clip.load("ViT-B/32", device=device)

# charger le Generator
G = Generator().to(device)
G.load_state_dict(torch.load('/kaggle/working/anime_cgan/checkpoints/G_epoch50.pth'))
G.eval()
print(" Modèles chargés")

In [ ]:
def generate_from_text(text, n_images=4):
    # encoder le texte avec CLIP
    tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_emb = model_clip.encode_text(tokens)  # (1, 512)
    
    # répéter l'embedding pour n_images
    text_emb = text_emb.repeat(n_images, 1).float()
    
    # générer
    z = torch.randn(n_images, 100).to(device)
    with torch.no_grad():
        fake_imgs = G(z, text_emb)
    
    # afficher
    fig, axes = plt.subplots(1, n_images, figsize=(12, 4))
    for i, ax in enumerate(axes):
        img = fake_imgs[i].permute(1, 2, 0).cpu().numpy()
        img = (img + 1) / 2  # dénormaliser [-1,1] → [0,1]
        img = img.clip(0, 1)
        ax.imshow(img)
        ax.axis('off')
    
    plt.suptitle(f'Prompt : "{text}"', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/anime_cgan/results/test_{text[:20].replace(" ","_")}.png')
    plt.show()

In [ ]:
prompts = [
    "girl with blue hair",
    "girl with red hair and smile",
    "boy with dark hair",
    "girl with green hair and sad expression"
]

for prompt in prompts:
    generate_from_text(prompt)
    print(f" Généré : {prompt}")

In [ ]:
G = Generator().to(device)
D = Discriminator().to(device)

opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0001, betas=(0.5, 0.999))

criterion = nn.BCELoss()

fixed_z = torch.randn(16, Z_DIM).to(device)
fixed_emb = dataset.embeddings[:16].to(device).float()

print(" Modèles réinitialisés")

In [ ]:
EPOCHS = 100
losses_G, losses_D = [], []

for epoch in range(EPOCHS):
    loss_G_epoch, loss_D_epoch = 0, 0

    for real_imgs, clip_embs in dataloader:
        real_imgs = real_imgs.to(device)
        clip_embs = clip_embs.to(device).float()
        batch_size = real_imgs.size(0)

        # label smoothing
        real_labels = torch.ones(batch_size, 1).to(device) * 0.9
        fake_labels = torch.zeros(batch_size, 1).to(device) + 0.1

        # --- Entraîner Discriminator (1x) ---
        z = torch.randn(batch_size, Z_DIM).to(device)
        fake_imgs = G(z, clip_embs)

        loss_D = criterion(D(real_imgs, clip_embs), real_labels) + \
                 criterion(D(fake_imgs.detach(), clip_embs), fake_labels)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Entraîner Generator (2x) ---
        for _ in range(2):
            z = torch.randn(batch_size, Z_DIM).to(device)
            fake_imgs = G(z, clip_embs)
            loss_G = criterion(D(fake_imgs, clip_embs), real_labels)
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        loss_G_epoch += loss_G.item()
        loss_D_epoch += loss_D.item()

    avg_G = loss_G_epoch / len(dataloader)
    avg_D = loss_D_epoch / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f}")

    # checkpoint + grille toutes les 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/G_epoch{epoch+1}.pth')
        torch.save(D.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/D_epoch{epoch+1}.pth')

        with torch.no_grad():
            fake_sample = G(fixed_z, fixed_emb)
        grid = vutils.make_grid(fake_sample, normalize=True, nrow=4)
        plt.figure(figsize=(8, 8))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis('off')
        plt.title(f'Epoch {epoch+1}')
        plt.savefig(f'/kaggle/working/anime_cgan/results/epoch_{epoch+1}.png')
        plt.show()

print(" Entraînement terminé !")

In [ ]:
def generate_from_text(text, n_images=4):
    # encoder le texte avec CLIP
    tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_emb = model_clip.encode_text(tokens)  # (1, 512)
    
    # répéter l'embedding pour n_images
    text_emb = text_emb.repeat(n_images, 1).float()
    
    # générer
    z = torch.randn(n_images, 100).to(device)
    with torch.no_grad():
        fake_imgs = G(z, text_emb)
    
    # afficher
    fig, axes = plt.subplots(1, n_images, figsize=(12, 4))
    for i, ax in enumerate(axes):
        img = fake_imgs[i].permute(1, 2, 0).cpu().numpy()
        img = (img + 1) / 2  # dénormaliser [-1,1] → [0,1]
        img = img.clip(0, 1)
        ax.imshow(img)
        ax.axis('off')
    
    plt.suptitle(f'Prompt : "{text}"', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/anime_cgan/results/test_{text[:20].replace(" ","_")}.png')
    plt.show()

In [ ]:
prompts = [
    "girl with blue hair",
    "girl with red hair and smile",
    "boy with dark hair",
    "girl with green hair and sad expression"
]

for prompt in prompts:
    generate_from_text(prompt)
    print(f" Généré : {prompt}")

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os, torch

class AnimeDataset128(Dataset):
    def __init__(self, img_dir, embeddings_path):
        data = torch.load(embeddings_path)
        self.embeddings = data['embeddings']
        self.filenames = data['filenames']
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),  # ← 128x128
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_dir, self.filenames[idx])).convert('RGB')
        return self.transform(img), self.embeddings[idx]

dataset = AnimeDataset128(
    img_dir='/kaggle/working/anime_cgan/data/images_5k',
    embeddings_path='/kaggle/working/anime_cgan/data/embeddings.pt'
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)  # batch 32 car plus lourd
print(f" Dataset 128x128 chargé : {len(dataset)} images")

In [ ]:
class Generator128(nn.Module):
    def __init__(self, z_dim=100, clip_dim=512, img_channels=3):
        super().__init__()
        
        self.fc = nn.Linear(z_dim + clip_dim, 512 * 4 * 4)
        
        self.net = nn.Sequential(
            # 4x4 → 8x8
            nn.ConvTranspose2d(512, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # 8x8 → 16x16
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 16x16 → 32x32
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 32x32 → 64x64
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 64x64 → 128x128  ← couche supplémentaire
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1),
            nn.Tanh()
        )
    
    def forward(self, z, clip_emb):
        x = torch.cat([z, clip_emb], dim=1)
        x = self.fc(x)
        x = x.view(-1, 512, 4, 4)
        return self.net(x)

In [ ]:
import torch.optim as optim
import torchvision.utils as vutils

Z_DIM = 100

opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0001, betas=(0.5, 0.999))
criterion = nn.BCELoss()

fixed_z = torch.randn(16, Z_DIM).to(device)
fixed_emb = dataset.embeddings[:16].to(device).float()

EPOCHS = 100
losses_G, losses_D = [], []

for epoch in range(EPOCHS):
    loss_G_epoch, loss_D_epoch = 0, 0

    for real_imgs, clip_embs in dataloader:
        real_imgs = real_imgs.to(device)
        clip_embs = clip_embs.to(device).float()
        batch_size = real_imgs.size(0)

        real_labels = torch.ones(batch_size, 1).to(device) * 0.9
        fake_labels = torch.zeros(batch_size, 1).to(device) + 0.1

        # --- Discriminator ---
        z = torch.randn(batch_size, Z_DIM).to(device)
        fake_imgs = G(z, clip_embs)
        loss_D = criterion(D(real_imgs, clip_embs), real_labels) + \
                 criterion(D(fake_imgs.detach(), clip_embs), fake_labels)
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Generator 2x ---
        for _ in range(2):
            z = torch.randn(batch_size, Z_DIM).to(device)
            fake_imgs = G(z, clip_embs)
            loss_G = criterion(D(fake_imgs, clip_embs), real_labels)
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        loss_G_epoch += loss_G.item()
        loss_D_epoch += loss_D.item()

    avg_G = loss_G_epoch / len(dataloader)
    avg_D = loss_D_epoch / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f}")

    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/G128_epoch{epoch+1}.pth')
        torch.save(D.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/D128_epoch{epoch+1}.pth')

        with torch.no_grad():
            fake_sample = G(fixed_z, fixed_emb)
        grid = vutils.make_grid(fake_sample, normalize=True, nrow=4)
        plt.figure(figsize=(10, 10))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis('off')
        plt.title(f'Epoch {epoch+1}')
        plt.savefig(f'/kaggle/working/anime_cgan/results/epoch128_{epoch+1}.png')
        plt.show()

print(" Entraînement terminé !")

In [ ]:
class Discriminator128(nn.Module):
    def __init__(self, clip_dim=512, img_channels=3):
        super().__init__()
        
        self.conv = nn.Sequential(
            # 128x128 → 64x64
            nn.Conv2d(img_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            # 64x64 → 32x32
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 32x32 → 16x16
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 16x16 → 8x8
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 8x8 → 4x4  ← couche supplémentaire
            nn.Conv2d(512, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        self.fc = nn.Sequential(
            nn.Linear(512 * 4 * 4 + clip_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, clip_emb):
        x = self.conv(img)
        x = x.view(x.size(0), -1)
        x = torch.cat([x, clip_emb], dim=1)
        return self.fc(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

G = Generator128().to(device)
D = Discriminator128().to(device)

z = torch.randn(4, 100).to(device)
emb = torch.randn(4, 512).to(device)
fake = G(z, emb)
score = D(fake, emb)

print(f"Generator output : {fake.shape}")   # (4, 3, 128, 128)
print(f" Discriminator output : {score.shape}")  # (4, 1)

In [ ]:
import torch.optim as optim
import torchvision.utils as vutils

Z_DIM = 100

opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0001, betas=(0.5, 0.999))
criterion = nn.BCELoss()

fixed_z = torch.randn(16, Z_DIM).to(device)
fixed_emb = dataset.embeddings[:16].to(device).float()

EPOCHS = 100
losses_G, losses_D = [], []

for epoch in range(EPOCHS):
    loss_G_epoch, loss_D_epoch = 0, 0

    for real_imgs, clip_embs in dataloader:
        real_imgs = real_imgs.to(device)
        clip_embs = clip_embs.to(device).float()
        batch_size = real_imgs.size(0)

        real_labels = torch.ones(batch_size, 1).to(device) * 0.9
        fake_labels = torch.zeros(batch_size, 1).to(device) + 0.1

        # --- Discriminator ---
        z = torch.randn(batch_size, Z_DIM).to(device)
        fake_imgs = G(z, clip_embs)
        loss_D = criterion(D(real_imgs, clip_embs), real_labels) + \
                 criterion(D(fake_imgs.detach(), clip_embs), fake_labels)
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Generator 2x ---
        for _ in range(2):
            z = torch.randn(batch_size, Z_DIM).to(device)
            fake_imgs = G(z, clip_embs)
            loss_G = criterion(D(fake_imgs, clip_embs), real_labels)
            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        loss_G_epoch += loss_G.item()
        loss_D_epoch += loss_D.item()

    avg_G = loss_G_epoch / len(dataloader)
    avg_D = loss_D_epoch / len(dataloader)
    losses_G.append(avg_G)
    losses_D.append(avg_D)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f}")

    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/G128_epoch{epoch+1}.pth')
        torch.save(D.state_dict(), f'/kaggle/working/anime_cgan/checkpoints/D128_epoch{epoch+1}.pth')

        with torch.no_grad():
            fake_sample = G(fixed_z, fixed_emb)
        grid = vutils.make_grid(fake_sample, normalize=True, nrow=4)
        plt.figure(figsize=(10, 10))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis('off')
        plt.title(f'Epoch {epoch+1}')
        plt.savefig(f'/kaggle/working/anime_cgan/results/epoch128_{epoch+1}.png')
        plt.show()

print("Entraînement terminé !")

In [ ]:
prompts = [
    "girl with blue hair",
    "girl with red hair and smile",
    "boy with dark hair",
    "girl with white hair",
    "girl with purple hair and sad expression",
    "girl with blonde hair"
]

for prompt in prompts:
    generate_from_text(prompt, n_images=4)
    print(f"✅ Généré : {prompt}")